# Oxford-IIIT Pet — test-set CAM audit

This notebook is the visual companion to the [full benchmark](../results/oxford_pet/RESULTS.md).
It shows **three frozen test examples per architecture** using the exact same
center-cropped input for CE, FCAL V1, FCAL V2, and—where available—Universal FCAL.

How to read each panel:

- green/red text means the predicted class is correct/incorrect;
- the white contour is the annotated foreground;
- `inside` is the share of positive **predicted-class CAM** mass inside that contour;
- CAM colors are normalized within each panel and use `jet` (blue = low,
  red = high).

The examples are selected reproducibly from saved test arrays: first a strong
localization comparison where CE and V2 are both correct, then a CE→V2
classification rescue when one exists, then another localization improvement.
This is a qualitative audit, not an accuracy estimate; use the full result tables
for quantitative claims.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

ASSETS = ROOT / "assets" / "result_showcases" / "oxford_pet"
manifest = pd.read_csv(ASSETS / "examples.csv")
print(f"Loaded {len(manifest)} frozen examples from {manifest['architecture'].nunique()} architectures")
manifest[["family", "architecture", "case_rank", "image_id", "true_class"]]

Loaded 18 frozen examples from 6 architectures


,family,architecture,case_rank,image_id,true_class
0,CNN,ResNet-50,1,3301,boxer
1,CNN,ResNet-50,2,6094,saint bernard
2,CNN,ResNet-50,3,3527,chihuahua
3,CNN,DenseNet-121,1,2519,american bulldog
4,CNN,DenseNet-121,2,346,Bengal
5,CNN,DenseNet-121,3,4295,great pyrenees
6,CNN,EfficientNet-B0,1,6553,scottish terrier
7,CNN,EfficientNet-B0,2,1766,Ragdoll
8,CNN,EfficientNet-B0,3,2371,Sphynx
9,ViT,ViT-Tiny/16,1,1768,Ragdoll


## CNNs

These CAMs are exact final-feature class activation maps: the classifier's own
linear weights projected back onto the final convolutional feature grid.

### ResNet-50

![ResNet-50 test cases](../assets/result_showcases/oxford_pet/resnet50_test_cases.png)

### DenseNet-121

![DenseNet-121 test cases](../assets/result_showcases/oxford_pet/densenet121_test_cases.png)

### EfficientNet-B0

![EfficientNet-B0 test cases](../assets/result_showcases/oxford_pet/efficientnet-b0_test_cases.png)

## Vision Transformers

The ViT runs use average-pooled patch tokens, so their class-specific patch CAMs
decompose the actual classifier score rather than substituting a generic
self-attention visualization.

### ViT-Tiny/16

![ViT-Tiny/16 test cases](../assets/result_showcases/oxford_pet/vit-tiny-p16-avg_test_cases.png)

### ViT-Small/16

![ViT-Small/16 test cases](../assets/result_showcases/oxford_pet/vit-small-p16-avg_test_cases.png)

### ViT-Base/16

![ViT-Base/16 test cases](../assets/result_showcases/oxford_pet/vit-base-p16-avg_test_cases.png)

## Selected-example audit table

The table below exposes every displayed prediction and inside-foreground value.
Because these examples were deliberately selected for visual diagnosis, do not
average this table or present it as benchmark performance.

In [2]:
display_columns = [
    column for column in manifest.columns
    if column in {"architecture", "case_rank", "image_id", "true_class"}
    or column.endswith("_correct")
    or column.endswith("_inside_fraction")
]
manifest[display_columns].style.format({
    column: "{:.1%}" for column in display_columns if column.endswith("_inside_fraction")
})

,architecture,case_rank,image_id,true_class,ce_correct,ce_inside_fraction,fcal_v1_correct,fcal_v1_inside_fraction,fcal_v2_correct,fcal_v2_inside_fraction,universal_correct,universal_inside_fraction
0,ResNet-50,1,3301,boxer,True,48.1%,True,51.9%,True,80.7%,True,56.1%
1,ResNet-50,2,6094,saint bernard,False,75.2%,True,73.2%,True,90.3%,False,78.6%
2,ResNet-50,3,3527,chihuahua,True,49.3%,True,55.2%,True,80.5%,True,57.5%
3,DenseNet-121,1,2519,american bulldog,True,72.7%,False,83.4%,True,92.4%,True,82.1%
4,DenseNet-121,2,346,Bengal,False,53.0%,True,53.3%,True,70.8%,False,59.8%
5,DenseNet-121,3,4295,great pyrenees,True,59.0%,True,62.3%,True,80.5%,True,64.2%
6,EfficientNet-B0,1,6553,scottish terrier,True,20.9%,False,30.5%,True,36.2%,True,26.4%
7,EfficientNet-B0,2,1766,Ragdoll,False,49.1%,True,46.8%,True,78.0%,True,46.3%
8,EfficientNet-B0,3,2371,Sphynx,True,55.0%,True,58.5%,True,71.5%,True,57.2%
9,ViT-Tiny/16,1,1768,Ragdoll,True,48.2%,True,50.7%,True,52.0%,True,48.8%


## Artifact provenance

The committed PNGs were reconstructed from the real best-validation checkpoints.
The accompanying `examples.csv` manifest freezes every displayed test ID,
prediction, correctness flag, confidence, and inside-foreground CAM value, so
this notebook remains auditable without downloading checkpoints or datasets.